<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Python profesional para analítica avanzada</p>
  </div>

</div>

# **GroupBy, Aggregations, Transform y Polars**

En este laboratorio estudiaremos uno de los patrones más importantes de la analítica tabular:

> **split → apply → combine**

Partiremos de Pandas y, para cada familia de operaciones relevante, construiremos después su equivalente idiomático en **Polars**.

No trataremos Polars como “Pandas con otra sintaxis”. Compararemos:

- modelo mental;
- API;
- expresiones;
- agregaciones;
- ventanas;
- UDFs;
- ejecución eager;
- ejecución lazy;
- rendimiento;
- equivalencia de resultados.

La secuencia será:

> **resolver en Pandas → interpretar → detectar límites → resolver en Polars → validar equivalencia → medir → interpretar**

## Objetivos

Al finalizar deberías poder:

1. Explicar el patrón split-apply-combine.
2. Utilizar `groupby` con una o varias claves.
3. Aplicar agregaciones simples y múltiples.
4. Utilizar named aggregations.
5. Diferenciar `agg`, `transform` y `apply`.
6. Entender por qué `apply` puede ser costoso.
7. Crear métricas relativas dentro de grupos.
8. Trabajar con rankings y ventanas.
9. Crear agregaciones condicionales.
10. Utilizar `group_by().agg()` en Polars.
11. Utilizar expresiones `pl.col(...)`.
12. Reproducir `transform` mediante `over`.
13. Evitar UDFs cuando existe una expresión nativa.
14. Diferenciar Polars eager y lazy.
15. Construir y ejecutar un plan lazy.
16. Comparar Pandas y Polars mediante benchmarks reproducibles.
17. Verificar que ambos motores producen resultados equivalentes.

## 1. Preparación del entorno

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

try:
    import polars as pl
    POLARS_DISPONIBLE = True
    print("Polars:", pl.__version__)
except ImportError:
    POLARS_DISPONIBLE = False
    print("Polars no está instalado.")
    print("Instalación recomendada: pip install polars")

print("Pandas:", pd.__version__)

> El notebook mantiene bloques protegidos para que pueda abrirse aunque Polars no esté instalado. Para completar la práctica comparativa debe instalarse Polars.

# 2. Dataset analítico

In [ ]:
N = 500_000

ventas = pd.DataFrame({
    "venta_id": np.arange(1, N + 1),
    "cliente_id": rng.integers(1, 50_001, N),
    "region": rng.choice(
        ["Norte", "Sur", "Este", "Oeste", "Centro"],
        N
    ),
    "canal": rng.choice(
        ["web", "tienda", "partner", "marketplace"],
        N,
        p=[0.45, 0.30, 0.15, 0.10]
    ),
    "producto": rng.choice(
        [f"P{i:02d}" for i in range(1, 31)],
        N
    ),
    "unidades": rng.integers(1, 9, N),
    "precio": rng.uniform(5, 250, N),
    "descuento": rng.choice(
        [0.0, 0.05, 0.10, 0.20],
        N,
        p=[0.55, 0.20, 0.18, 0.07]
    ),
    "fecha": pd.Timestamp("2025-01-01")
        + pd.to_timedelta(
            rng.integers(0, 365 * 24 * 60, N),
            unit="m"
        )
})

ventas["importe"] = (
    ventas["unidades"]
    * ventas["precio"]
    * (1 - ventas["descuento"])
)

ventas.head()

## 2.1 Auditoría mínima antes de agrupar

In [ ]:
pd.DataFrame({
    "dtype": ventas.dtypes.astype(str),
    "nulos": ventas.isna().sum(),
    "unicos": ventas.nunique()
})

### Interpretación

Las columnas `region`, `canal` y `producto` tienen baja cardinalidad y son candidatas naturales para agrupar.

Antes de cualquier agregación debemos conocer:

- granularidad de las filas;
- significado de las claves;
- variables numéricas;
- posibles nulos.

# 3. ¿Qué hace realmente un `groupby`?

Conceptualmente:

1. **split**: divide las filas según una o varias claves;
2. **apply**: calcula una operación en cada grupo;
3. **combine**: combina los resultados.

Empezaremos con una sola clave.

In [ ]:
ventas.groupby("region")["importe"].sum()

El resultado responde a una pregunta concreta:

> ¿Cuál es la facturación acumulada de cada región?

## 3.1 Convertir el resultado en tabla

In [ ]:
facturacion_region = (
    ventas.groupby(
        "region",
        as_index=False
    )
    .agg(
        facturacion=("importe", "sum")
    )
)

facturacion_region

## 3.2 Visualizar

In [ ]:
orden = facturacion_region.sort_values("facturacion")

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(
    orden["region"],
    orden["facturacion"]
)
ax.set(
    title="Facturación total por región",
    xlabel="Facturación",
    ylabel="Región"
)
plt.show()

### Interpretación

El gráfico permite comparar magnitudes absolutas entre regiones. Como los datos fueron generados con regiones aproximadamente equiprobables, no esperamos diferencias estructurales extremas.

# 4. Agrupar por varias claves

In [ ]:
region_canal = (
    ventas.groupby(
        ["region", "canal"],
        as_index=False
    )
    .agg(
        facturacion=("importe", "sum")
    )
)

region_canal.head(12)

## 4.1 Visualización matricial

In [ ]:
matriz_region_canal = (
    region_canal
    .pivot(
        index="region",
        columns="canal",
        values="facturacion"
    )
)

matriz_region_canal

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

imagen = ax.imshow(
    matriz_region_canal.values,
    aspect="auto"
)

ax.set_xticks(
    range(len(matriz_region_canal.columns)),
    matriz_region_canal.columns
)
ax.set_yticks(
    range(len(matriz_region_canal.index)),
    matriz_region_canal.index
)

ax.set_title("Facturación: región × canal")

for i in range(matriz_region_canal.shape[0]):
    for j in range(matriz_region_canal.shape[1]):
        ax.text(
            j,
            i,
            f"{matriz_region_canal.iloc[i, j] / 1e6:.1f}M",
            ha="center",
            va="center"
        )

plt.colorbar(imagen, ax=ax, label="Facturación")
plt.show()

### Interpretación

La agregación multidimensional permite detectar combinaciones región-canal con mayor o menor volumen.

Esto es diferente de estudiar cada dimensión por separado.

# 5. Varias agregaciones

In [ ]:
resumen_region = (
    ventas.groupby("region")
    .agg({
        "importe": ["sum", "mean", "median", "max"],
        "unidades": ["sum", "mean"],
        "cliente_id": "nunique"
    })
)

resumen_region

### Problema observado

La sintaxis anterior funciona, pero genera columnas jerárquicas (`MultiIndex`) y los nombres resultantes no son especialmente cómodos para pipelines posteriores.

## 5.1 Observar las columnas resultantes

In [ ]:
resumen_region.columns

# 6. Named aggregations

In [ ]:
resumen_region_named = (
    ventas.groupby(
        "region",
        as_index=False
    )
    .agg(
        facturacion=("importe", "sum"),
        ticket_medio=("importe", "mean"),
        ticket_mediano=("importe", "median"),
        ticket_maximo=("importe", "max"),
        unidades=("unidades", "sum"),
        unidades_medias=("unidades", "mean"),
        clientes_unicos=("cliente_id", "nunique"),
        operaciones=("venta_id", "size")
    )
)

resumen_region_named

### Interpretación

Las named aggregations hacen explícita la relación:

`nombre_salida = (columna_entrada, función)`

Esto produce una tabla plana y autodescriptiva.

# 7. Métricas derivadas después de agrupar

In [ ]:
resumen_region_named["facturacion_por_cliente"] = (
    resumen_region_named["facturacion"]
    / resumen_region_named["clientes_unicos"]
)

resumen_region_named["operaciones_por_cliente"] = (
    resumen_region_named["operaciones"]
    / resumen_region_named["clientes_unicos"]
)

resumen_region_named

Es frecuente combinar agregaciones con métricas derivadas. Conviene distinguir qué métricas se calculan a nivel fila y cuáles a nivel agregado.

# 8. Error conceptual: media de medias

Una inconsistencia analítica habitual consiste en promediar medias de grupos sin considerar su tamaño.

Vamos a construir un ejemplo donde se vea claramente.

In [ ]:
ejemplo_medias = pd.DataFrame({
    "grupo": ["A"] * 2 + ["B"] * 8,
    "valor": [100, 100] + [0] * 8
})

ejemplo_medias

In [ ]:
medias_grupo = (
    ejemplo_medias
    .groupby("grupo")["valor"]
    .mean()
)

media_de_medias = medias_grupo.mean()
media_global = ejemplo_medias["valor"].mean()

print("Media de medias:", media_de_medias)
print("Media global:", media_global)

### Interpretación

La media de medias da el mismo peso a cada grupo, independientemente de cuántas observaciones contiene.

Antes de agregar debemos tener claro qué estadístico queremos representar.

# 9. `size()` vs `count()`

In [ ]:
demo_nulos = pd.DataFrame({
    "grupo": ["A", "A", "B", "B"],
    "valor": [10, np.nan, 20, np.nan]
})

print("size:")
display(
    demo_nulos.groupby("grupo").size()
)

print("count:")
display(
    demo_nulos.groupby("grupo")["valor"].count()
)

### Diferencia

- `size()` cuenta filas.
- `count()` cuenta valores no nulos.

La diferencia es crítica cuando existen missing values.

# 10. `transform`: mantener granularidad original

In [ ]:
ventas["media_importe_region"] = (
    ventas.groupby("region")["importe"]
    .transform("mean")
)

ventas[[
    "venta_id",
    "region",
    "importe",
    "media_importe_region"
]].head()

`transform` devuelve una Serie alineada con las filas originales.

Esto permite incorporar estadísticas de grupo sin reducir la granularidad.

## 10.1 Diferencia respecto a `agg`

In [ ]:
print(
    "Filas originales:",
    len(ventas)
)

print(
    "Filas tras agg:",
    len(
        ventas.groupby("region")
        .agg(media=("importe", "mean"))
    )
)

print(
    "Filas tras transform:",
    len(
        ventas.groupby("region")["importe"]
        .transform("mean")
    )
)

# 11. Desviación respecto al grupo

In [ ]:
ventas["desviacion_region"] = (
    ventas["importe"]
    - ventas["media_importe_region"]
)

ventas[[
    "region",
    "importe",
    "media_importe_region",
    "desviacion_region"
]].head()

## 11.1 Validación

In [ ]:
validacion_desviacion = (
    ventas.groupby("region")["desviacion_region"]
    .mean()
)

validacion_desviacion

### Interpretación

La media de las desviaciones respecto a la media del grupo debería estar muy próxima a cero.

# 12. Porcentaje sobre el total del grupo

In [ ]:
ventas["facturacion_region"] = (
    ventas.groupby("region")["importe"]
    .transform("sum")
)

ventas["peso_en_region"] = (
    ventas["importe"]
    / ventas["facturacion_region"]
)

ventas[[
    "region",
    "importe",
    "facturacion_region",
    "peso_en_region"
]].head()

## 12.1 Validar que suma 1 por región

In [ ]:
ventas.groupby("region")["peso_en_region"].sum()

Esta validación comprueba una propiedad matemática esperada del indicador.

# 13. Estandarización dentro de grupo

In [ ]:
media_region = (
    ventas.groupby("region")["importe"]
    .transform("mean")
)

std_region = (
    ventas.groupby("region")["importe"]
    .transform("std")
)

ventas["z_importe_region"] = (
    (ventas["importe"] - media_region)
    / std_region
)

ventas[[
    "region",
    "importe",
    "z_importe_region"
]].head()

## 13.1 Visualizar z-scores

In [ ]:
muestra_z = ventas.sample(
    10_000,
    random_state=SEED
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(
    muestra_z["z_importe_region"],
    bins=50
)
ax.set(
    title="Distribución del importe estandarizado dentro de región",
    xlabel="z-score",
    ylabel="Frecuencia"
)
plt.show()

### Interpretación

El z-score indica cuántas desviaciones estándar se encuentra una operación respecto al comportamiento de su propia región.

# 14. Ranking dentro de grupo

In [ ]:
ventas["ranking_importe_region"] = (
    ventas.groupby("region")["importe"]
    .rank(
        method="dense",
        ascending=False
    )
)

ventas.loc[
    ventas["ranking_importe_region"] <= 3,
    [
        "region",
        "venta_id",
        "importe",
        "ranking_importe_region"
    ]
].sort_values(
    ["region", "ranking_importe_region"]
).head(20)

### Interpretación

El ranking se calcula dentro de cada partición de `region`. Este patrón tiene un equivalente natural mediante funciones ventana en Polars.

# 15. Agregación condicional

In [ ]:
ventas["es_venta_alta"] = (
    ventas["importe"] >= 1000
)

resumen_altas = (
    ventas.groupby(
        "region",
        as_index=False
    )
    .agg(
        operaciones=("venta_id", "size"),
        ventas_altas=("es_venta_alta", "sum"),
        pct_altas=("es_venta_alta", "mean")
    )
)

resumen_altas

## 15.1 Visualizar porcentaje

In [ ]:
orden_altas = resumen_altas.sort_values("pct_altas")

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(
    orden_altas["region"],
    orden_altas["pct_altas"]
)
ax.set(
    title="Proporción de ventas de importe ≥ 1000",
    xlabel="Proporción",
    ylabel="Región"
)
plt.show()

# 16. `apply`: primero un caso funcional

In [ ]:
def estadisticas_personalizadas(grupo):
    return pd.Series({
        "p10": grupo["importe"].quantile(0.10),
        "p90": grupo["importe"].quantile(0.90),
        "rango_p80": (
            grupo["importe"].quantile(0.90)
            - grupo["importe"].quantile(0.10)
        )
    })

resultado_apply = (
    ventas.groupby("region")
    .apply(
        estadisticas_personalizadas,
        include_groups=False
    )
)

resultado_apply

### Problema potencial

`apply` es muy flexible, pero esa flexibilidad puede introducir ejecución Python grupo a grupo.

Antes de utilizarlo debemos comprobar si la misma operación puede expresarse mediante agregaciones nativas.

# 17. Sustituir `apply` por agregaciones nativas

In [ ]:
resultado_nativo = (
    ventas.groupby(
        "region",
        as_index=False
    )
    .agg(
        p10=("importe", lambda s: s.quantile(0.10)),
        p90=("importe", lambda s: s.quantile(0.90))
    )
)

resultado_nativo["rango_p80"] = (
    resultado_nativo["p90"]
    - resultado_nativo["p10"]
)

resultado_nativo

Todavía aparecen lambdas para cuantiles, pero hemos evitado devolver manualmente una Serie por grupo. Más adelante compararemos esto con expresiones nativas de Polars.

# 18. Benchmark `apply` vs agregación

In [ ]:
def medir(funcion, repeticiones=5):
    tiempos = []

    for _ in range(repeticiones):
        inicio = time.perf_counter()
        funcion()
        tiempos.append(
            time.perf_counter() - inicio
        )

    return np.array(tiempos)


def ejecutar_apply():
    return (
        ventas.groupby("region")
        .apply(
            estadisticas_personalizadas,
            include_groups=False
        )
    )


def ejecutar_agg():
    resultado = (
        ventas.groupby(
            "region",
            as_index=False
        )
        .agg(
            p10=("importe", lambda s: s.quantile(0.10)),
            p90=("importe", lambda s: s.quantile(0.90))
        )
    )

    resultado["rango_p80"] = (
        resultado["p90"]
        - resultado["p10"]
    )

    return resultado


t_apply = medir(ejecutar_apply)
t_agg = medir(ejecutar_agg)

benchmark_apply = pd.DataFrame({
    "metodo": ["apply", "agg"],
    "media_ms": [
        t_apply.mean() * 1000,
        t_agg.mean() * 1000
    ],
    "std_ms": [
        t_apply.std() * 1000,
        t_agg.std() * 1000
    ]
})

benchmark_apply

## 18.1 Visualizar benchmark

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.bar(
    benchmark_apply["metodo"],
    benchmark_apply["media_ms"],
    yerr=benchmark_apply["std_ms"],
    capsize=4
)

ax.set(
    title="Pandas: apply vs agg",
    ylabel="Tiempo medio (ms)"
)

plt.show()

### Interpretación

La conclusión relevante es metodológica:

> si una operación puede escribirse mediante primitivas vectorizadas o agregaciones nativas, conviene probar esa alternativa antes de recurrir a UDFs.

# 19. Preparar Polars

In [ ]:
if POLARS_DISPONIBLE:
    ventas_pl = pl.from_pandas(
        ventas[[
            "venta_id",
            "cliente_id",
            "region",
            "canal",
            "producto",
            "unidades",
            "precio",
            "descuento",
            "fecha",
            "importe"
        ]]
    )

    print(ventas_pl.schema)
    print(ventas_pl.head())
else:
    print("Bloque omitido: instalar polars.")

## Diferencia conceptual importante

Polars no utiliza un índice de filas como Pandas. El trabajo habitual se expresa mediante:

- `.select()`;
- `.filter()`;
- `.with_columns()`;
- `.group_by()`;
- expresiones `pl.col(...)`.

Esto evita trasladar mecánicamente patrones como `.loc` o `reset_index()`.

# 20. Primer `group_by` en Polars

In [ ]:
if POLARS_DISPONIBLE:
    facturacion_region_pl = (
        ventas_pl
        .group_by("region")
        .agg(
            pl.col("importe")
            .sum()
            .alias("facturacion")
        )
        .sort("region")
    )

    print(facturacion_region_pl)

### Comparación sintáctica

Pandas:

```python
df.groupby("region", as_index=False).agg(
    facturacion=("importe", "sum")
)
```

Polars:

```python
df.group_by("region").agg(
    pl.col("importe").sum().alias("facturacion")
)
```

En Polars la agregación se expresa como una **expresión sobre columnas**.

# 21. Varias agregaciones en Polars

In [ ]:
if POLARS_DISPONIBLE:
    resumen_region_pl = (
        ventas_pl
        .group_by("region")
        .agg(
            pl.col("importe")
            .sum()
            .alias("facturacion"),

            pl.col("importe")
            .mean()
            .alias("ticket_medio"),

            pl.col("importe")
            .median()
            .alias("ticket_mediano"),

            pl.col("importe")
            .max()
            .alias("ticket_maximo"),

            pl.col("unidades")
            .sum()
            .alias("unidades"),

            pl.col("cliente_id")
            .n_unique()
            .alias("clientes_unicos"),

            pl.len()
            .alias("operaciones")
        )
        .sort("region")
    )

    print(resumen_region_pl)

### Interpretación

El sistema de expresiones permite definir múltiples cálculos dentro de una sola operación de agregación.

# 22. Agregación condicional en Polars

In [ ]:
if POLARS_DISPONIBLE:
    altas_pl = (
        ventas_pl
        .group_by("region")
        .agg(
            pl.len().alias("operaciones"),

            (
                pl.col("importe") >= 1000
            )
            .sum()
            .alias("ventas_altas"),

            (
                pl.col("importe") >= 1000
            )
            .mean()
            .alias("pct_altas")
        )
        .sort("region")
    )

    print(altas_pl)

Aquí no es necesario crear previamente una columna booleana permanente: la condición puede vivir dentro de la expresión de agregación.

# 23. Equivalente de `transform`: ventanas con `over`

In [ ]:
if POLARS_DISPONIBLE:
    ventas_transform_pl = (
        ventas_pl
        .with_columns(
            pl.col("importe")
            .mean()
            .over("region")
            .alias("media_importe_region"),

            pl.col("importe")
            .sum()
            .over("region")
            .alias("facturacion_region")
        )
        .with_columns(
            (
                pl.col("importe")
                - pl.col("media_importe_region")
            )
            .alias("desviacion_region"),

            (
                pl.col("importe")
                / pl.col("facturacion_region")
            )
            .alias("peso_en_region")
        )
    )

    print(
        ventas_transform_pl.select(
            "venta_id",
            "region",
            "importe",
            "media_importe_region",
            "desviacion_region",
            "peso_en_region"
        ).head()
    )

### Interpretación

`Expr.over("region")` aplica una expresión como ventana por grupo y mantiene la granularidad original.

Es conceptualmente el equivalente más cercano a muchos usos de `groupby(...).transform(...)` de Pandas.

# 24. Validar pesos en Polars

In [ ]:
if POLARS_DISPONIBLE:
    validar_peso_pl = (
        ventas_transform_pl
        .group_by("region")
        .agg(
            pl.col("peso_en_region")
            .sum()
            .alias("suma_pesos")
        )
        .sort("region")
    )

    print(validar_peso_pl)

# 25. Ranking dentro de grupo en Polars

In [ ]:
if POLARS_DISPONIBLE:
    ranking_pl = (
        ventas_pl
        .with_columns(
            pl.col("importe")
            .rank(
                method="dense",
                descending=True
            )
            .over("region")
            .alias("ranking_importe_region")
        )
        .filter(
            pl.col("ranking_importe_region") <= 3
        )
        .sort(
            ["region", "ranking_importe_region"]
        )
    )

    print(
        ranking_pl.select(
            "region",
            "venta_id",
            "importe",
            "ranking_importe_region"
        ).head(20)
    )

# 26. Cuantiles en Polars sin UDF Python

In [ ]:
if POLARS_DISPONIBLE:
    percentiles_pl = (
        ventas_pl
        .group_by("region")
        .agg(
            pl.col("importe")
            .quantile(0.10)
            .alias("p10"),

            pl.col("importe")
            .quantile(0.90)
            .alias("p90")
        )
        .with_columns(
            (
                pl.col("p90")
                - pl.col("p10")
            )
            .alias("rango_p80")
        )
        .sort("region")
    )

    print(percentiles_pl)

### Lección importante

Antes de usar una función Python personalizada en Polars hay que buscar si existe una **expresión nativa**.

Las expresiones nativas permiten al motor razonar sobre el plan y optimizarlo.

# 27. UDF en Polars: ejemplo deliberadamente menos idiomático

In [ ]:
if POLARS_DISPONIBLE:
    ejemplo_udf = (
        ventas_pl
        .select(
            pl.col("importe")
            .map_elements(
                lambda x: x * 1.21,
                return_dtype=pl.Float64
            )
            .alias("importe_con_factor")
        )
        .head()
    )

    print(ejemplo_udf)

## 27.1 Reescribir con expresión nativa

In [ ]:
if POLARS_DISPONIBLE:
    ejemplo_nativo = (
        ventas_pl
        .select(
            (
                pl.col("importe") * 1.21
            )
            .alias("importe_con_factor")
        )
        .head()
    )

    print(ejemplo_nativo)

La segunda versión comunica la operación al motor de Polars, en lugar de tratar cada elemento mediante una función Python.

# 28. Eager vs Lazy

En modo **eager**, cada operación se ejecuta inmediatamente.

En modo **lazy**, Polars construye un plan y lo ejecuta al llamar a `.collect()`.

Esto permite optimizaciones globales de la consulta.

In [ ]:
if POLARS_DISPONIBLE:
    consulta_lazy = (
        ventas_pl
        .lazy()
        .filter(
            pl.col("importe") >= 500
        )
        .group_by(
            ["region", "canal"]
        )
        .agg(
            pl.col("importe")
            .sum()
            .alias("facturacion"),

            pl.len()
            .alias("operaciones")
        )
        .sort(
            ["region", "canal"]
        )
    )

    print(consulta_lazy)

## 28.1 Ejecutar con `collect()`

In [ ]:
if POLARS_DISPONIBLE:
    resultado_lazy = (
        consulta_lazy
        .collect()
    )

    print(resultado_lazy.head(12))

# 29. Inspeccionar el plan lazy

In [ ]:
if POLARS_DISPONIBLE:
    print(
        consulta_lazy.explain(
            optimized=True
        )
    )

### Interpretación

En lazy mode, Polars puede reorganizar el plan antes de ejecutarlo. Esto es especialmente relevante cuando se trabaja con lectura desde disco mediante operaciones como `scan_csv`.

# 30. Demostración con CSV y `scan_csv`

In [ ]:
ruta_csv = Path("ventas_polars_demo.csv")

ventas[[
    "venta_id",
    "region",
    "canal",
    "importe",
    "unidades"
]].to_csv(
    ruta_csv,
    index=False
)

print(
    "CSV creado:",
    ruta_csv.resolve(),
    "MB:",
    round(
        ruta_csv.stat().st_size / 1024**2,
        2
    )
)

In [ ]:
if POLARS_DISPONIBLE:
    consulta_scan = (
        pl.scan_csv(
            ruta_csv
        )
        .filter(
            pl.col("importe") >= 500
        )
        .group_by("region")
        .agg(
            pl.col("importe")
            .sum()
            .alias("facturacion")
        )
    )

    print(
        consulta_scan.explain(
            optimized=True
        )
    )

### Qué queremos observar

Aunque el CSV contenga varias columnas, el plan puede identificar qué columnas son necesarias para la consulta.

Esta optimización no existe de la misma forma en el modelo eager clásico de Pandas.

# 31. Benchmark Pandas vs Polars: groupby sencillo

In [ ]:
def pandas_groupby():
    return (
        ventas.groupby(
            "region",
            as_index=False
        )
        .agg(
            facturacion=("importe", "sum"),
            ticket_medio=("importe", "mean"),
            operaciones=("venta_id", "size")
        )
    )


def polars_groupby():
    if not POLARS_DISPONIBLE:
        return None

    return (
        ventas_pl
        .group_by("region")
        .agg(
            pl.col("importe")
            .sum()
            .alias("facturacion"),

            pl.col("importe")
            .mean()
            .alias("ticket_medio"),

            pl.len()
            .alias("operaciones")
        )
    )


t_pd = medir(
    pandas_groupby,
    repeticiones=7
)

if POLARS_DISPONIBLE:
    t_pl = medir(
        polars_groupby,
        repeticiones=7
    )
else:
    t_pl = np.array([np.nan])

benchmark_motores = pd.DataFrame({
    "motor": ["Pandas", "Polars"],
    "media_ms": [
        t_pd.mean() * 1000,
        t_pl.mean() * 1000
    ],
    "std_ms": [
        t_pd.std() * 1000,
        t_pl.std() * 1000
    ]
})

benchmark_motores

## 31.1 Visualizar benchmark

In [ ]:
if POLARS_DISPONIBLE:
    fig, ax = plt.subplots(figsize=(6, 4))

    ax.bar(
        benchmark_motores["motor"],
        benchmark_motores["media_ms"],
        yerr=benchmark_motores["std_ms"],
        capsize=4
    )

    ax.set(
        title="GroupBy: Pandas vs Polars",
        ylabel="Tiempo medio (ms)"
    )

    plt.show()
else:
    print("Instalar Polars para ejecutar la comparación.")

### Interpretación

El benchmark se realiza sobre el mismo dataset y la misma pregunta analítica.

No debemos generalizar una única medición como “Polars siempre es X veces más rápido”. El resultado depende de:

- operación;
- tamaño;
- tipos;
- hardware;
- versión;
- coste de conversión;
- eager/lazy.

# 32. Benchmark transformación de grupo

In [ ]:
def pandas_transform():
    return (
        ventas.groupby("region")["importe"]
        .transform("mean")
    )


def polars_window():
    if not POLARS_DISPONIBLE:
        return None

    return (
        ventas_pl
        .select(
            pl.col("importe")
            .mean()
            .over("region")
            .alias("media_region")
        )
    )


t_pd_transform = medir(
    pandas_transform,
    repeticiones=7
)

if POLARS_DISPONIBLE:
    t_pl_window = medir(
        polars_window,
        repeticiones=7
    )
else:
    t_pl_window = np.array([np.nan])

benchmark_window = pd.DataFrame({
    "motor": ["Pandas transform", "Polars over"],
    "media_ms": [
        t_pd_transform.mean() * 1000,
        t_pl_window.mean() * 1000
    ]
})

benchmark_window

## 32.1 Visualizar

In [ ]:
if POLARS_DISPONIBLE:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(
        benchmark_window["motor"],
        benchmark_window["media_ms"]
    )
    ax.set(
        title="Operación por ventana/grupo",
        ylabel="Tiempo medio (ms)"
    )
    ax.tick_params(axis="x", rotation=15)
    plt.show()

# 33. Validación de equivalencia Pandas vs Polars

In [ ]:
resultado_pd = (
    pandas_groupby()
    .sort_values("region")
    .reset_index(drop=True)
)

if POLARS_DISPONIBLE:
    resultado_pl_pd = (
        polars_groupby()
        .sort("region")
        .to_pandas()
    )

    comparacion = resultado_pd.merge(
        resultado_pl_pd,
        on="region",
        suffixes=("_pd", "_pl")
    )

    comparacion["dif_facturacion"] = (
        comparacion["facturacion_pd"]
        - comparacion["facturacion_pl"]
    )

    comparacion["dif_ticket"] = (
        comparacion["ticket_medio_pd"]
        - comparacion["ticket_medio_pl"]
    )

    comparacion
else:
    print("Polars no disponible.")

## 33.1 Validación numérica

In [ ]:
if POLARS_DISPONIBLE:
    print(
        "Facturación equivalente:",
        np.allclose(
            comparacion["facturacion_pd"],
            comparacion["facturacion_pl"]
        )
    )

    print(
        "Ticket equivalente:",
        np.allclose(
            comparacion["ticket_medio_pd"],
            comparacion["ticket_medio_pl"]
        )
    )

    print(
        "Operaciones equivalentes:",
        (
            comparacion["operaciones_pd"]
            ==
            comparacion["operaciones_pl"]
        ).all()
    )

### Regla del curso

No aceptaremos una migración a Polars únicamente porque sea más rápida.

Debe demostrarse también que conserva la semántica y los resultados requeridos.

# 34. Medir el coste de conversión Pandas → Polars

In [ ]:
if POLARS_DISPONIBLE:
    tiempos_conversion = []

    for _ in range(5):
        inicio = time.perf_counter()

        _ = pl.from_pandas(
            ventas[[
                "venta_id",
                "region",
                "importe"
            ]]
        )

        tiempos_conversion.append(
            time.perf_counter()
            - inicio
        )

    print(
        "Conversión media:",
        round(
            np.mean(tiempos_conversion) * 1000,
            3
        ),
        "ms"
    )

### Interpretación

Si un pipeline convierte repetidamente de Pandas a Polars y viceversa, el coste de conversión puede reducir parte de la ventaja obtenida.

Una migración profesional debe considerar el pipeline completo, no una operación aislada.

# 35. Comparación de modelos mentales

| Necesidad | Pandas | Polars |
|---|---|---|
| Seleccionar columnas | `df[...]` / `.loc` | `.select()` |
| Filtrar filas | máscara + `.loc` | `.filter()` |
| Crear columnas | asignación / `.assign()` | `.with_columns()` |
| Agrupar | `.groupby()` | `.group_by()` |
| Transformación por grupo | `.transform()` | expresión `.over()` |
| UDF | `.apply()` | `map_elements` / `map_groups` cuando sea inevitable |
| Ejecución | eager | eager + lazy |
| Índice | sí | no |
| Optimización global del plan | limitada | lazy query optimizer |

La intención no es aprender una traducción línea por línea, sino reconocer que Polars favorece un **modelo basado en expresiones**.

# 36. Mini-práctica integrada Pandas → Polars

Construiremos el mismo análisis en ambos motores.

Pregunta:

> Por región y canal, calcular facturación, operaciones, clientes únicos, ticket medio y porcentaje de operaciones superiores a 750 €.

## 36.1 Pandas

In [ ]:
analisis_pd = (
    ventas.assign(
        superior_750=ventas["importe"] > 750
    )
    .groupby(
        ["region", "canal"],
        as_index=False
    )
    .agg(
        facturacion=("importe", "sum"),
        operaciones=("venta_id", "size"),
        clientes_unicos=("cliente_id", "nunique"),
        ticket_medio=("importe", "mean"),
        pct_superior_750=("superior_750", "mean")
    )
    .sort_values(
        ["region", "canal"]
    )
    .reset_index(drop=True)
)

analisis_pd.head(12)

## 36.2 Polars eager

In [ ]:
if POLARS_DISPONIBLE:
    analisis_pl = (
        ventas_pl
        .group_by(
            ["region", "canal"]
        )
        .agg(
            pl.col("importe")
            .sum()
            .alias("facturacion"),

            pl.len()
            .alias("operaciones"),

            pl.col("cliente_id")
            .n_unique()
            .alias("clientes_unicos"),

            pl.col("importe")
            .mean()
            .alias("ticket_medio"),

            (
                pl.col("importe") > 750
            )
            .mean()
            .alias("pct_superior_750")
        )
        .sort(
            ["region", "canal"]
        )
    )

    print(analisis_pl.head(12))

## 36.3 Polars lazy

In [ ]:
if POLARS_DISPONIBLE:
    analisis_lazy = (
        ventas_pl
        .lazy()
        .group_by(
            ["region", "canal"]
        )
        .agg(
            pl.col("importe")
            .sum()
            .alias("facturacion"),

            pl.len()
            .alias("operaciones"),

            pl.col("cliente_id")
            .n_unique()
            .alias("clientes_unicos"),

            pl.col("importe")
            .mean()
            .alias("ticket_medio"),

            (
                pl.col("importe") > 750
            )
            .mean()
            .alias("pct_superior_750")
        )
        .sort(
            ["region", "canal"]
        )
    )

    print(
        analisis_lazy
        .collect()
        .head(12)
    )

## 36.4 Validación cruzada

In [ ]:
if POLARS_DISPONIBLE:
    analisis_pl_pd = (
        analisis_pl
        .to_pandas()
        .reset_index(drop=True)
    )

    columnas_numericas = [
        "facturacion",
        "operaciones",
        "clientes_unicos",
        "ticket_medio",
        "pct_superior_750"
    ]

    resultados_equivalentes = {}

    for columna in columnas_numericas:
        resultados_equivalentes[columna] = np.allclose(
            analisis_pd[columna],
            analisis_pl_pd[columna]
        )

    resultados_equivalentes

## 36.5 Visualización analítica

In [ ]:
matriz_pct = (
    analisis_pd
    .pivot(
        index="region",
        columns="canal",
        values="pct_superior_750"
    )
)

fig, ax = plt.subplots(figsize=(8, 5))

im = ax.imshow(
    matriz_pct.values,
    aspect="auto",
    vmin=0,
    vmax=1
)

ax.set_xticks(
    range(len(matriz_pct.columns)),
    matriz_pct.columns
)
ax.set_yticks(
    range(len(matriz_pct.index)),
    matriz_pct.index
)

ax.set_title(
    "Proporción de operaciones > 750 €"
)

for i in range(matriz_pct.shape[0]):
    for j in range(matriz_pct.shape[1]):
        ax.text(
            j,
            i,
            f"{matriz_pct.iloc[i, j]:.1%}",
            ha="center",
            va="center"
        )

plt.colorbar(
    im,
    ax=ax,
    label="Proporción"
)

plt.show()

### Interpretación

La visualización final es independiente del motor utilizado para obtener la tabla analítica.

Esto permite separar dos decisiones:

- **qué pregunta analítica queremos responder**;
- **qué motor ejecuta mejor el pipeline**.

# 37. Limpieza de fichero temporal

In [ ]:
ruta_csv.unlink(
    missing_ok=True
)

print("Fichero temporal eliminado.")

# 38. Checklist GroupBy + Polars

- [ ] ¿Conozco la granularidad original?
- [ ] ¿Sé qué significa cada clave de agrupación?
- [ ] ¿Distingo `size` y `count`?
- [ ] ¿Uso named aggregations?
- [ ] ¿Evito media de medias no ponderada accidental?
- [ ] ¿Sé cuándo necesito `agg`?
- [ ] ¿Sé cuándo necesito mantener granularidad con `transform`?
- [ ] ¿Valido propiedades esperadas del resultado?
- [ ] ¿Evito `apply` si existe una operación nativa?
- [ ] ¿Entiendo `group_by().agg()` en Polars?
- [ ] ¿Uso expresiones Polars?
- [ ] ¿Sé reproducir transformaciones mediante `.over()`?
- [ ] ¿Evito UDFs Python en Polars cuando existe expresión nativa?
- [ ] ¿Distingo eager y lazy?
- [ ] ¿Sé cuándo se ejecuta `.collect()`?
- [ ] ¿Valido equivalencia Pandas/Polars?
- [ ] ¿Mido antes de comparar rendimiento?
- [ ] ¿Incluyo el coste de conversión en la decisión?

# 39. Ejercicios finales

## Ejercicio 1 — GroupBy básico
Calcula facturación, operaciones y clientes únicos por región.

## Ejercicio 2 — Dos dimensiones
Repite el análisis por región y canal y represéntalo.

## Ejercicio 3 — Named aggregations
Genera al menos ocho indicadores con nombres explícitos.

## Ejercicio 4 — `size` vs `count`
Introduce nulos y demuestra la diferencia.

## Ejercicio 5 — Error analítico
Construye un ejemplo donde la media de medias sea diferente de la media global.

## Ejercicio 6 — `transform`
Calcula la media del grupo, desviación y peso porcentual de cada fila.

## Ejercicio 7 — Ventanas
Calcula ranking de las cinco operaciones más grandes dentro de cada región.

## Ejercicio 8 — `apply`
Resuelve una operación mediante `apply`, mide su coste y busca una alternativa nativa.

## Ejercicio 9 — Polars básico
Reproduce tres agregaciones Pandas mediante `group_by().agg()`.

## Ejercicio 10 — Polars window
Reproduce un `transform` de Pandas mediante `.over()`.

## Ejercicio 11 — UDF
Escribe una operación con `map_elements` y después sustitúyela por una expresión nativa.

## Ejercicio 12 — Lazy
Construye una consulta lazy con filtro, group_by y agregación. Inspecciona el plan y ejecuta con `.collect()`.

## Ejercicio 13 — CSV lazy
Guarda un dataset y utiliza `scan_csv()` para construir una consulta.

## Ejercicio 14 — Benchmark
Compara Pandas y Polars en al menos dos operaciones, con varias repeticiones.

## Ejercicio 15 — Equivalencia
Comprueba numéricamente que ambos motores devuelven resultados equivalentes.

## Ejercicio 16 — Coste de migración
Mide Pandas → Polars y Polars → Pandas. Discute cuándo ese coste importa.

## Ejercicio 17 — Reto final
Sobre al menos 500.000 filas:

1. crea un análisis con tres claves;
2. genera mínimo ocho KPIs;
3. incluye una métrica condicional;
4. incluye una ventana;
5. implementa Pandas;
6. implementa Polars eager;
7. implementa Polars lazy;
8. valida resultados;
9. realiza benchmarks;
10. visualiza e interpreta;
11. justifica qué motor elegirías.

# 40. Preguntas de reflexión

1. ¿Qué significa split-apply-combine?
2. ¿Cuándo `agg` reduce granularidad?
3. ¿Por qué `transform` conserva filas?
4. ¿Qué diferencia hay entre `size` y `count`?
5. ¿Por qué la media de medias puede ser incorrecta?
6. ¿Por qué `apply` debe utilizarse con criterio?
7. ¿Qué es una expresión en Polars?
8. ¿Qué sustituye a muchos usos de `transform` en Polars?
9. ¿Por qué Polars no necesita un índice como Pandas?
10. ¿Qué diferencia hay entre eager y lazy?
11. ¿Qué ocurre al llamar a `.collect()`?
12. ¿Por qué una UDF puede limitar optimizaciones?
13. ¿Polars será siempre más rápido?
14. ¿Por qué debemos medir el coste de conversión?
15. ¿Qué significa validar equivalencia entre motores?